# Portfolio Optimization Example

This notebook demonstrates portfolio optimization techniques including:
- Maximum Sharpe Ratio
- Minimum Variance
- Efficient Frontier

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from market_risk_hub.data.market_data import MarketDataFetcher
from market_risk_hub.portfolio.analytics import PortfolioAnalytics
from market_risk_hub.utils.visualization import RiskVisualizer

## 1. Fetch Market Data

In [ ]:
fetcher = MarketDataFetcher()
tickers = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'GLD', 'TLT']

data = fetcher.get_market_data(tickers, period='5y')
returns = data['returns']

print(f"Data loaded: {len(returns)} observations")
print(f"Assets: {', '.join(returns.columns)}")

## 2. Current Equal-Weight Portfolio Analysis

In [ ]:
# Equal weights
equal_weights = np.array([1/len(tickers)] * len(tickers))

portfolio = PortfolioAnalytics(returns, equal_weights)
summary = portfolio.get_summary(risk_free_rate=0.04)

print("Equal-Weight Portfolio Metrics:")
print("=" * 50)
print(summary)

## 3. Correlation Analysis

In [ ]:
correlation_matrix = portfolio.calculate_correlation_matrix()

print("Correlation Matrix:")
print(correlation_matrix)

fig = RiskVisualizer.plot_correlation_heatmap(correlation_matrix)
fig.show()

## 4. Maximum Sharpe Ratio Portfolio

In [ ]:
# Optimize for maximum Sharpe ratio
portfolio_opt = PortfolioAnalytics(returns)
optimal_sharpe = portfolio_opt.optimize_sharpe_ratio(risk_free_rate=0.04)

print("Maximum Sharpe Ratio Portfolio:")
print("=" * 50)
print(f"Expected Return: {optimal_sharpe['return']:.2%}")
print(f"Volatility: {optimal_sharpe['volatility']:.2%}")
print(f"Sharpe Ratio: {optimal_sharpe['sharpe_ratio']:.4f}")
print("\nOptimal Weights:")
print(optimal_sharpe['weights'])

## 5. Minimum Variance Portfolio

In [ ]:
# Optimize for minimum variance
min_var = portfolio_opt.optimize_minimum_variance()

print("Minimum Variance Portfolio:")
print("=" * 50)
print(f"Expected Return: {min_var['return']:.2%}")
print(f"Volatility: {min_var['volatility']:.2%}")
print("\nOptimal Weights:")
print(min_var['weights'])

## 6. Efficient Frontier

In [ ]:
# Calculate efficient frontier
ef_returns, ef_vols, ef_sharpes = portfolio_opt.efficient_frontier(
    n_portfolios=100,
    risk_free_rate=0.04
)

# Current portfolio metrics
current_metrics = {
    'return': summary['annualized_return'],
    'volatility': summary['annualized_volatility']
}

# Visualize
fig = RiskVisualizer.plot_efficient_frontier(
    ef_returns,
    ef_vols,
    ef_sharpes,
    current_metrics
)
fig.show()

## 7. Portfolio Comparison

In [ ]:
# Compare portfolios
comparison = pd.DataFrame({
    'Equal Weight': [
        summary['annualized_return'],
        summary['annualized_volatility'],
        summary['sharpe_ratio']
    ],
    'Max Sharpe': [
        optimal_sharpe['return'],
        optimal_sharpe['volatility'],
        optimal_sharpe['sharpe_ratio']
    ],
    'Min Variance': [
        min_var['return'],
        min_var['volatility'],
        (min_var['return'] - 0.04) / min_var['volatility']
    ]
}, index=['Return', 'Volatility', 'Sharpe Ratio'])

print("Portfolio Comparison:")
print("=" * 50)
print(comparison)

## 8. Weight Allocation Comparison

In [ ]:
weights_comparison = pd.DataFrame({
    'Equal Weight': equal_weights,
    'Max Sharpe': optimal_sharpe['weights'].values,
    'Min Variance': min_var['weights'].values
}, index=tickers)

print("Weight Allocation Comparison:")
print("=" * 50)
print(weights_comparison)